### Multilingual Text RAG

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

c:\Users\USER\rag_course\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the NLLB Model

In [2]:
# model_name = 'facebook/nllb-200-distilled-600M'

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# print('NLLB model loaded')
# # print(f'   Model: {model_name}')

In [3]:
# print('✅ NLLB model loaded')
# print(f'   Model: {model_name}')
# print(f'   Tokenizer: {type(tokenizer).__name__}')
# print(f'   Model class: {type(model).__name__}')

Write the NLLB translate function

In [4]:
# # NLLB uses FLORES-200 language codes
# NLLB_LANGS = {
#     'English':         'eng_Latn',
#     'Yoruba':          'yor_Latn',
#     'Hausa':           'hau_Latn',
#     'Igbo':            'ibo_Latn',
#     'Nigerian Pidgin': 'pcm_Latn',
# }


# # The translate function
# def translation_nllb(text: str, target_lang: str, source_lang: str = 'English') -> str:
#     '''
#     Translate text between Nigerian languages using NLLB-200.
#     text:        the text to translate
#     target_lang: 'Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin', or 'English'
#     source_lang: defaults to 'English'
#     '''
    
#     # 1. Map display names -> FLORES-200 codes
#     src_code = NLLB_LANGS[source_lang]
#     tgt_code = NLLB_LANGS[target_lang]
    
#     # 2. Translate lang using tokenizer
#     tokenizer.src_lang = src_code
    
#     # 3. Tokenize the input text
#     inputs = tokenizer(text, return_tensors='pt')
    
#     # 4. Generate translation tokens
#     with torch.no_grad():
#         generated = model.generate(
#             **inputs,
#             forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code),
#             max_length=512
#         )
        
#     # 5. Decode tokens back into text
#     return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

# print('translate_nllb ready')

STEP 4 — Test it

In [5]:
# english = 'To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.'

# print(f'🇬🇧 English: {english}')
# print()
# print(f'🇳🇬 Yoruba: {translation_nllb(english, "Yoruba")}')
# print()
# print(f'🇳🇬 Hausa: {translation_nllb(english, "Hausa")}')
# print(f'🇳🇬 Igbo: {translation_nllb(english, "Igbo")}')
# print()
# print(f'🇳🇬 Pidgin: {translation_nllb(english, "Nigerian Pidgin")}')

In [6]:
load_dotenv()

# OpenAI client — for translation
client = OpenAI()

# LangChain LLM — for RAG
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Embeddings for the vector store
embeddings = OpenAIEmbeddings()

Load the Vector Store and Build the RAG Chain

In [7]:
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print(f'Vector store loaded — {vectorstore._collection.count()} chunks')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded — 82 chunks


translate() function

In [8]:
SUPPORTED_LANGS = ['English', 'Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin']


def translate(text: str, target_lang: str) -> str:
    '''
    Translate English text to a Nigerian language using GPT-4o-mini.

    target_lang: 'Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin', or 'English'
    '''
    
    # If already in the target language, return as-is
    if target_lang == 'English':
        return text
    
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following English text to {target_lang}.\n'
        f'\n'
        f'RULES:\n'
        f'1. Output ONLY {target_lang} — no English mixed in, except proper nouns.\n'
        f'2. Use the correct tone marks and diacritics for {target_lang}.\n'
        f'3. Keep agricultural terms accurate and natural.\n'
        f'4. Match the tone of voice a farmer would actually use.\n'
        f'5. Output ONLY the translation. No explanations, no quotes.\n'
    )
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': text}
        ],
        temperature=0,
    )
    
    return response.choices[0].message.content.strip()

print('✅ translate() ready')
print(f'   Supported: {SUPPORTED_LANGS}')

✅ translate() ready
   Supported: ['English', 'Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin']


Test

In [9]:
english = (
    'To treat cassava mosaic disease, use disease-free cuttings, '
    'plant resistant varieties, and remove infected plants early.'
)

print('🇬🇧 English:', english)
print()
for lang in ['Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin']:
    print(f'🇳🇬 {lang:<18}: {translate(english, lang)}')
    print()

🇬🇧 English: To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.

🇳🇬 Yoruba            : Láti tọju àrùn àkàrà, lo ìkòkò tí kò ní àrùn, gbin irú ẹ̀yà tí ó ní ìdáhùn, kí o sì yọ àwọn ọgbin tó ní àrùn kúrò níbẹ̀ ní kánkán.

🇳🇬 Hausa             : Don magance cutar mosaic na akamfa, yi amfani da yankan da ba su da cuta, shuka nau'in da ke jurewa, kuma cire shukar da ta kamu da cutar tun da wuri.

🇳🇬 Igbo              : Iji wụpụ ọrịa mosaic nke ọka, jiri mkpụrụ osisi na-enweghị ọrịa, tinye ụdị ndị na-eguzogide, ma wepụ osisi ndị nwere ọrịa n'oge.

🇳🇬 Nigerian Pidgin   : To treat cassava mosaic disease, use cuttings wey no get disease, plant resistant varieties, and remove infected plants early.



The RAG Chain

In [10]:
# rag_prompt = ChatPromptTemplate.from_messages([
#     ('system', 'Answer using ONLY the context below. If the answer is not in the context, say "I don\'t know".'),
#     ('human', 'Context:\n{context}\n\nQuery: {query}'),
# ])

# rag_chain = rag_prompt | llm | StrOutputParser()

# def rag_answer(question: str) -> str:
#     '''
#     Retrieve English docs and generate an English answer.
#     '''
    
#     docs = retriever.invoke(question)
#     context = '\n\n'.join(doc.page_content for doc in docs)
    
#     return rag_chain.invoke({
#         'context': context,
#         'query': question
#     })
    
# print('RAG chain ready')

Test the RAG Chain in English

In [11]:
# answer = rag_answer('How do I treat cassava mosaic disease?')
# print(answer)

The Translation Function

In [12]:
# def translate(text: str, target_lang: str) -> str:
#     '''
#     Translate English text to the target language using GPT-4o-mini.

#     target_lang: 'Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin', or 'English'
#     '''
    
    
    

In [13]:
# def translate(text: str, target_lang: str) -> str:
#     """Translate English to a Nigerian language with strict constraints."""
#     system = (
#         f'Translate the following text to {target_lang}.\n'
#         f'\n'
#         f'RULES:\n'
#         f'1. Use ONLY {target_lang} script and Latin letters — no Chinese or other scripts.\n'
#         f'2. Do NOT leave English words except proper nouns (names, brands).\n'
#         f'3. If unsure of a technical term, use the closest common {target_lang} word.\n'
#         f'4. Output ONLY the translation. No explanations.\n'
#     )

#     response = client.chat.completions.create(
#         model='gpt-4o-mini',
#         messages=[
#             {'role': 'system', 'content': system},
#             {'role': 'user', 'content': text},
#         ],
#         temperature=0,
#     )
#     return response.choices[0].message.content.strip()


# print('✅ stricter translate() ready')

In [14]:
# english = 'To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.'

# print('🇬🇧 English:', english)
# print()
# print('🇳🇬 Yoruba: ', translate(english, 'Yoruba'))
# print()
# print('🇳🇬 Hausa:  ', translate(english, 'Hausa'))
# print()
# print('🇳🇬 Igbo:   ', translate(english, 'Igbo'))
# print()
# print('🇳🇬 Pidgin: ', translate(english, 'Nigerian Pidgin'))

Building the production-grade translator with terminology in Chroma.

STEP A — Store terminology in Chroma


In [15]:
from langchain_core.documents import Document

terminology_docs = [
    # Crops
    Document(
        page_content='cassava | Yoruba: ẹgẹ́ | Hausa: rogo | Igbo: akpụ | Pidgin: cassava',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='maize | Yoruba: àgbàdo | Hausa: masara | Igbo: ọka | Pidgin: maize',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='yam | Yoruba: iṣu | Hausa: doya | Igbo: ji | Pidgin: yam',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='rice | Yoruba: ìrẹsì | Hausa: shinkafa | Igbo: osikapa | Pidgin: rice',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='tomato | Yoruba: tòmátì | Hausa: tumatir | Igbo: tomatọ | Pidgin: tomato',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='pepper | Yoruba: ata | Hausa: barkono | Igbo: ose | Pidgin: pepper',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='groundnut | Yoruba: èpà | Hausa: gyada | Igbo: ahụekere | Pidgin: groundnut',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),
    Document(
        page_content='cocoa | Yoruba: koko | Hausa: koko | Igbo: koko | Pidgin: cocoa',
        metadata={'type': 'terminology', 'category': 'crop', 'source': 'IITA'},
    ),

    # Diseases (kept in English — technical terms)
    Document(
        page_content='cassava mosaic disease | Yoruba: àrùn mosaic ẹgẹ́ | Hausa: cutar mosaic na rogo | Igbo: ọrịa mosaic akpụ',
        metadata={'type': 'terminology', 'category': 'disease', 'source': 'IITA'},
    ),
    Document(
        page_content='late blight | Yoruba: àrùn blight | Hausa: cutar blight | Igbo: ọrịa blight',
        metadata={'type': 'terminology', 'category': 'disease', 'source': 'IITA'},
    ),
    Document(
        page_content='fall armyworm | Yoruba: kòkòrò ogun | Hausa: tsutsar soja | Igbo: igurube agha',
        metadata={'type': 'terminology', 'category': 'pest', 'source': 'IITA'},
    ),
]

print(f'✅ {len(terminology_docs)} terminology docs prepared')
for d in terminology_docs:
    print(f'   [{d.metadata["category"]:8}] {d.page_content.split(" | ")[0]}')


✅ 11 terminology docs prepared
   [crop    ] cassava
   [crop    ] maize
   [crop    ] yam
   [crop    ] rice
   [crop    ] tomato
   [crop    ] pepper
   [crop    ] groundnut
   [crop    ] cocoa
   [disease ] cassava mosaic disease
   [disease ] late blight
   [pest    ] fall armyworm


STEP B — Store them in Chroma


In [16]:
TERMINOLOGY_DIR = r'C:\Users\USER\rag_course\chroma_db_terminology'

terminology_store = Chroma.from_documents(
    documents=terminology_docs,
    embedding=embeddings,
    collection_name='terminology',
    persist_directory=TERMINOLOGY_DIR
)

terminology_retriever = terminology_store.as_retriever(search_kwargs={'k': 5})

print(f'✅ Terminology store ready — {terminology_store._collection.count()} docs')
print(f'   Persisted to: {TERMINOLOGY_DIR}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Terminology store ready — 33 docs
   Persisted to: C:\Users\USER\rag_course\chroma_db_terminology


STEP C — Test retrieval

In [17]:
for query in [
    'How do I treat cassava mosaic disease?',
    'Best time to plant maize?',
    'Fall armyworm damage on maize',
    'My yam tubers are rotting',
]:
    print(f'🔍 Query: {query}')
    docs = terminology_retriever.invoke(query)
    for d in docs:
        print(f'   • {d.page_content}')
    print()

🔍 Query: How do I treat cassava mosaic disease?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


   • cassava mosaic disease | Yoruba: àrùn mosaic ẹgẹ́ | Hausa: cutar mosaic na rogo | Igbo: ọrịa mosaic akpụ
   • cassava mosaic disease | Yoruba: àrùn mosaic ẹgẹ́ | Hausa: cutar mosaic na rogo | Igbo: ọrịa mosaic akpụ
   • cassava mosaic disease | Yoruba: àrùn mosaic ẹgẹ́ | Hausa: cutar mosaic na rogo | Igbo: ọrịa mosaic akpụ
   • cassava | Yoruba: ẹgẹ́ | Hausa: rogo | Igbo: akpụ | Pidgin: cassava
   • cassava | Yoruba: ẹgẹ́ | Hausa: rogo | Igbo: akpụ | Pidgin: cassava

🔍 Query: Best time to plant maize?
   • maize | Yoruba: àgbàdo | Hausa: masara | Igbo: ọka | Pidgin: maize
   • maize | Yoruba: àgbàdo | Hausa: masara | Igbo: ọka | Pidgin: maize
   • maize | Yoruba: àgbàdo | Hausa: masara | Igbo: ọka | Pidgin: maize
   • fall armyworm | Yoruba: kòkòrò ogun | Hausa: tsutsar soja | Igbo: igurube agha
   • fall armyworm | Yoruba: kòkòrò ogun | Hausa: tsutsar soja | Igbo: igurube agha

🔍 Query: Fall armyworm damage on maize
   • fall armyworm | Yoruba: kòkòrò ogun | Hausa: tsutsar soja |

STEP D — The production translate function

In [18]:
# All crop names we know about — used to find them in English input
KNOWN_CROP_NAMES = {
    'cassava', 'maize', 'yam', 'rice', 'tomato',
    'pepper', 'groundnut', 'cocoa', 'cocoyam', 'plantain',
    'banana', 'sorghum', 'millet', 'beans', 'onion',
}


def translate(text: str, target_lang: str) -> str:
    '''
    Translate English to a Nigerian language.
    Terminology from Chroma.
    Unknown crops stay in English — enforced.
    '''
    if target_lang == 'English':
        return text

    # 1. Retrieve relevant terminology
    terms = get_relevant_terms(text)

    # 2. Find crops in the text that are NOT in terminology → force English
    text_lower = text.lower()
    missing_crops = [
        crop for crop in KNOWN_CROP_NAMES
        if crop in text_lower and crop not in terms.lower()
    ]

    # 3. Build the prompt
    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following English text to {target_lang}.\n'
        f'\n'
        f'MANDATORY VOCABULARY (use these exact words):\n'
        f'{terms}\n'
        f'\n'
    )

    if missing_crops:
        system += (
            f'DO NOT TRANSLATE THESE ENGLISH WORDS — leave them exactly as-is:\n'
            + '\n'.join(f'  - {c}' for c in missing_crops)
            + '\n\n'
        )

    system += (
        f'RULES:\n'
        f'1. Output ONLY {target_lang} — no English mixed in, EXCEPT the words above.\n'
        f'2. Use the correct tone marks and diacritics.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print('✅ translate() v3 ready — enforces English fallback')

✅ translate() v3 ready — enforces English fallback


In [19]:
def get_relevant_terms(text: str) -> str:
    '''
    Retrieve terminology relevant to this text from Chroma.
    '''
    docs = terminology_retriever.invoke(text)
    return '\n'.join(f'- {d.page_content}' for d in docs)


print('✅ get_relevant_terms ready')

✅ get_relevant_terms ready


STEP E — Test it

In [20]:
english = 'How do I store cocoyam after harvest?'

print('🇬🇧 English:', english)
print()
for lang in ['Yoruba', 'Hausa', 'Igbo', 'Nigerian Pidgin']:
    print(f'🇳🇬 {lang:<18}: {translate(english, lang)}')
    print()

print('─' * 60)
print('Make sure cassava still works:')
print()

english2 = 'To treat cassava mosaic disease, use disease-free cuttings.'
for lang in ['Yoruba', 'Hausa', 'Igbo']:
    print(f'🇳🇬 {lang:<18}: {translate(english2, lang)}')

🇬🇧 English: How do I store cocoyam after harvest?

🇳🇬 Yoruba            : Báwo ni mo ṣe le tọju cocoyam lẹ́yìn ikore?

🇳🇬 Hausa             : Yaya zan adana rogo bayan girbi?

🇳🇬 Igbo              : Olee otú m ga-esi chekwaa cocoyam mgbe m nwetara ya?

🇳🇬 Nigerian Pidgin   : How I go fit store cocoyam after harvest?

────────────────────────────────────────────────────────────
Make sure cassava still works:

🇳🇬 Yoruba            : Láti tọju àrùn mosaic ẹgẹ́, lo ìkòkò tí kò ní àrùn.
🇳🇬 Hausa             : Don magance cutar mosaic na rogo, yi amfani da yankan da ba su da cuta.
🇳🇬 Igbo              : Iji wụnye ọrịa mosaic akpụ, jiri mkpụrụ osisi na-enweghị ọrịa.


In [21]:
# Diagnostics — what's loaded?
needed = ['client', 'embeddings', 'terminology_store',
          'terminology_retriever', 'get_relevant_terms',
          'KNOWN_CROP_NAMES', 'translate']

for name in needed:
    status = '✅' if name in globals() else '❌ MISSING'
    print(f'{status}  {name}')

✅  client
✅  embeddings
✅  terminology_store
✅  terminology_retriever
✅  get_relevant_terms
✅  KNOWN_CROP_NAMES
✅  translate


In [22]:
def quality_check(original: str, translated: str, target_lang: str) -> tuple[bool, str]:
    '''
    Return (ok, reason). If ok is False, reason explains why.
    '''
    # 1. Empty output?
    if not translated or not translated.strip():
        return False, 'empty output'

    # 2. Only punctuation / whitespace?
    stripped = translated.strip()
    if all(not c.isalnum() for c in stripped):
        return False, 'no alphanumeric content'

    # 3. Length ratio check — translated should be in a reasonable range
    ratio = len(translated) / max(len(original), 1)
    if ratio < 0.4:
        return False, f'too short (ratio={ratio:.2f})'
    if ratio > 4.0:
        return False, f'too long (ratio={ratio:.2f})'

    # 4. Too much English left over? (for non-Pidgin targets)
    if target_lang != 'Nigerian Pidgin':
        english_words = {'the', 'and', 'of', 'to', 'for', 'is', 'are', 'was', 'with', 'how'}
        words = translated.lower().split()
        if words:
            english_count = sum(1 for w in words if w in english_words)
            if english_count / len(words) > 0.3:
                return False, 'too much English remaining'

    # 5. Known-bad pattern: very few unique chars (like NLLB Pidgin bug)
    if len(set(translated)) < 5:
        return False, 'insufficient character variety'

    return True, 'ok'


print('✅ quality_check ready')

✅ quality_check ready


In [23]:
def translate_safe(text: str, target_lang: str, max_retries: int = 2) -> str:
    '''
    Translate with a quality check and retry.
    If all retries fail, return the English original with a marker.
    '''
    if target_lang == 'English':
        return text

    for attempt in range(max_retries + 1):
        result = translate(text, target_lang)
        ok, reason = quality_check(text, result, target_lang)
        if ok:
            return result
        print(f'⚠️  Bad translation (attempt {attempt + 1}): {reason}')

    print(f'❌ All retries failed for {target_lang} — returning English')
    return f'[EN] {text}'


print('✅ translate_safe ready')

✅ translate_safe ready


In [24]:
import json
import hashlib
from pathlib import Path

CACHE_FILE = Path(r'C:\Users\USER\rag_course\translation_cache.json')
cache = json.loads(CACHE_FILE.read_text(encoding='utf-8')) if CACHE_FILE.exists() else {}


def cache_key(text: str, target_lang: str) -> str:
    return hashlib.sha256(f'{target_lang}|{text}'.encode()).hexdigest()[:16]


def translate_cached(text: str, target_lang: str) -> str:
    '''
    Full production translator:
      cache → terminology RAG → translate → quality check → cache
    '''
    if target_lang == 'English':
        return text

    key = cache_key(text, target_lang)
    if key in cache:
        return cache[key]

    result = translate_safe(text, target_lang)

    cache[key] = result
    CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding='utf-8')
    return result


print('✅ translate_cached ready')
print(f'   Cache file: {CACHE_FILE}')
print(f'   Entries: {len(cache)}')

✅ translate_cached ready
   Cache file: C:\Users\USER\rag_course\translation_cache.json
   Entries: 0


In [25]:
import time

text = 'To treat cassava mosaic disease, use disease-free cuttings.'

# First call — hits GPT
t0 = time.time()
result1 = translate_cached(text, 'Yoruba')
t1 = time.time()
print(f'First call  ({t1-t0:.2f}s): {result1}')

# Second call — hits cache
t0 = time.time()
result2 = translate_cached(text, 'Yoruba')
t1 = time.time()
print(f'Cached call ({t1-t0:.4f}s): {result2}')

print()
print(f'Cache entries now: {len(cache)}')

First call  (4.19s): Láti tọju àrùn mosaic ẹgẹ́, lo ìkòkò tí kò ní àrùn.
Cached call (0.0000s): Láti tọju àrùn mosaic ẹgẹ́, lo ìkòkò tí kò ní àrùn.

Cache entries now: 1


Build the English RAG chain

In [32]:
rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are AgriVoice, a farming assistant for Nigerian farmers.\n'
     'Answer using ONLY the context below. Be concise — the answer '
     'will be spoken aloud, so keep it under 3 sentences.\n'
     'If the answer is not in the context, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])

rag_chain = rag_prompt | llm | StrOutputParser()

def rag_answer(question: str) -> str:
    '''
    Retrieve relevant docs and generate an English answer.
    '''
    
    docs = retriever.invoke(question)
    context = '\n\n'.join(doc.page_content for doc in docs)
    
    return rag_chain.invoke({
        'context': context,
        'question': question
    })
    

Test in English

In [33]:
answer = rag_answer('How do I treat cassava mosaic disease?')
print('❓ How do I treat cassava mosaic disease?')
print(f'💬 {answer}')
print()

answer = rag_answer('What is the capital of France?')
print('❓ What is the capital of France?')
print(f'💬 {answer}')

❓ How do I treat cassava mosaic disease?
💬 Use disease-free cuttings for planting, choose resistant cassava varieties, and remove infected plants immediately. Control whitefly vectors with appropriate insecticides and avoid planting near infected fields.

❓ What is the capital of France?
💬 I don't know.


The full multilingual pipeline

In [34]:
def multilingual_rag(question: str, language: str) -> dict:
    '''
    Full pipeline:
      1. Translate question → English
      2. RAG answer in English
      3. Translate answer back → user's language

    Returns dict with every step visible for debugging.
    '''
    # 1. Translate question to English
    if language == 'English':
        question_en = question
    else:
        question_en = translate(question, 'English')  # won't work — see note below
    # ... see STEP below

Add reverse translation

In [35]:
def translate_to_english(text: str, source_lang: str) -> str:
    '''
    Translate FROM a Nigerian language TO English.
    '''
    if source_lang == 'English':
        return text

    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following {source_lang} text to English.\n'
        f'\n'
        f'RULES:\n'
        f'1. Output ONLY English.\n'
        f'2. Keep agricultural terms accurate.\n'
        f'3. Output ONLY the translation. No explanations.\n'
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print('✅ translate_to_english ready')

✅ translate_to_english ready


The full pipeline

In [36]:
def multilingual_rag(question: str, language: str) -> dict:
    '''
    Full multilingual RAG pipeline.

    question: text in the user's language
    language: 'English', 'Yoruba', 'Hausa', 'Igbo', or 'Nigerian Pidgin'
    '''
    # 1. Question → English
    question_en = translate_to_english(question, language) if language != 'English' else question

    # 2. RAG answer in English
    answer_en = rag_answer(question_en)

    # 3. Answer → user's language
    answer_local = translate_cached(answer_en, language) if language != 'English' else answer_en

    return {
        'question_original': question,
        'question_english':  question_en,
        'answer_english':    answer_en,
        'answer_local':      answer_local,
        'language':          language,
    }


print('✅ multilingual_rag ready')

✅ multilingual_rag ready


Test the full pipeline

In [37]:
# Test in English
print('══ Test 1: English ══')
r = multilingual_rag('How do I treat cassava mosaic disease?', 'English')
print(f'Q: {r["question_original"]}')
print(f'A: {r["answer_local"]}')
print()

# Test in Yoruba — we'll ask in Yoruba
print('══ Test 2: Yoruba ══')
r = multilingual_rag('Báwo ni mo ṣe lè tọ́jú àrùn mosaic ẹgẹ́?', 'Yoruba')
print(f'Q (original): {r["question_original"]}')
print(f'Q (English):  {r["question_english"]}')
print(f'A (English):  {r["answer_english"]}')
print(f'A (Yoruba):   {r["answer_local"]}')
print()

# Test in Hausa
print('══ Test 3: Hausa ══')
r = multilingual_rag('Yaya zan magance cutar mosaic na rogo?', 'Hausa')
print(f'Q (original): {r["question_original"]}')
print(f'Q (English):  {r["question_english"]}')
print(f'A (English):  {r["answer_english"]}')
print(f'A (Hausa):    {r["answer_local"]}')

══ Test 1: English ══
Q: How do I treat cassava mosaic disease?
A: Use disease-free cuttings for planting, choose resistant cassava varieties, and remove and destroy infected plants immediately. Control whitefly vectors with appropriate insecticides and avoid planting near infected fields.

══ Test 2: Yoruba ══
Q (original): Báwo ni mo ṣe lè tọ́jú àrùn mosaic ẹgẹ́?
Q (English):  How can I manage mosaic disease in cassava?
A (English):  To manage mosaic disease in cassava, use disease-free cuttings for planting, choose resistant varieties, and remove and destroy infected plants immediately. Additionally, do not plant cassava close to infected fields to prevent the spread of the disease. Control whitefly vectors with appropriate insecticides.
A (Yoruba):   Lati ṣakoso àrùn mosaic ẹgẹ́ ninu ẹgẹ́, lo awọn gige ti ko ni àrùn fun ikore, yan awọn irugbin ti o ni idena, ki o si yọkuro ki o si pa awọn ọgbin ti o ni àrùn lẹsẹkẹsẹ. Pẹlupẹlu, ma ṣe gbin ẹgẹ́ nitosi awọn ilẹ ti o ni àrùn lati dena 

In [38]:
def translate_to_english(text: str, source_lang: str) -> str:
    '''
    Translate FROM a Nigerian language TO English.
    Uses terminology from Chroma so crop names reverse correctly.
    '''
    if source_lang == 'English':
        return text

    # Retrieve terminology relevant to this text
    terms = get_relevant_terms(text)

    system = (
        f'You are an expert translator for Nigerian languages.\n'
        f'Translate the following {source_lang} text to English.\n'
        f'\n'
        f'MANDATORY VOCABULARY — use these to decode crop/disease names:\n'
        f'{terms}\n'
        f'\n'
        f'RULES:\n'
        f'1. Output ONLY English.\n'
        f'2. Use the terminology above to correctly identify crops/diseases.\n'
        f'3. Keep agricultural terms accurate.\n'
        f'4. Output ONLY the translation. No explanations.\n'
    )

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': text},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print('✅ translate_to_english v2 ready — uses terminology')

✅ translate_to_english v2 ready — uses terminology


In [39]:
extra_terminology = [
    Document(
        page_content='whitefly | Yoruba: kòkòrò funfun | Hausa: farar kwari | Igbo: ijiji ọcha',
        metadata={'type': 'terminology', 'category': 'pest', 'source': 'IITA'},
    ),
    Document(
        page_content='aphids | Yoruba: kòkòrò kékeré | Hausa: ƙananan kwari | Igbo: obere ahụhụ',
        metadata={'type': 'terminology', 'category': 'pest', 'source': 'IITA'},
    ),
    Document(
        page_content='stem borer | Yoruba: kòkòrò igi | Hausa: tsutsar kara | Igbo: igurube osisi',
        metadata={'type': 'terminology', 'category': 'pest', 'source': 'IITA'},
    ),
]

terminology_store.add_documents(extra_terminology)

print(f'✅ Added {len(extra_terminology)} pests')
print(f'   Terminology store now has {terminology_store._collection.count()} docs')

✅ Added 3 pests
   Terminology store now has 36 docs


In [40]:
print('══ Yoruba ══')
r = multilingual_rag('Báwo ni mo ṣe lè tọ́jú àrùn mosaic ẹgẹ́?', 'Yoruba')
print(f'Q (English):  {r["question_english"]}')
print(f'A (Yoruba):   {r["answer_local"]}')
print()

print('══ Hausa ══')
r = multilingual_rag('Yaya zan magance cutar mosaic na rogo?', 'Hausa')
print(f'Q (English):  {r["question_english"]}')
print(f'A (Hausa):    {r["answer_local"]}')
print()

print('══ Igbo ══')
r = multilingual_rag('Kedu ka m ga-esi gwọọ ọrịa mosaic akpụ?', 'Igbo')
print(f'Q (English):  {r["question_english"]}')
print(f'A (Igbo):     {r["answer_local"]}')

══ Yoruba ══
Q (English):  How can I manage cassava mosaic disease?
A (Yoruba):   Lati ṣakoso àrùn mosaic ẹgẹ́, lo awọn gige ti ko ni àrùn fun ikore, yan awọn irugbin ẹgẹ́ ti o ni idena, ki o si yọ awọn ọgbin ti o ni àrùn lẹsẹkẹsẹ. Pẹlupẹlu, ṣakoso kòkòrò funfun pẹlu awọn insecticides to yẹ ki o si yago fun ikore nitosi awọn ilẹ ti o ni àrùn.

══ Hausa ══
Q (English):  How can I control cassava mosaic disease?
A (Hausa):    Don sarrafa cutar mosaic na rogo, yi amfani da itatuwa marasa cuta don shuka, zaɓi nau'ikan rogo masu juriya, kuma cire shukar da ta kamu da cuta nan da nan. Bugu da ƙari, sarrafa farar kwari tare da magungunan kwari masu dacewa kuma ka guji shuka kusa da filayen da suka kamu da cuta.

══ Igbo ══
Q (English):  How can I treat cassava mosaic disease?
A (Igbo):     Iji wụnye ọrịa mosaic akpụ, jiri mkpụrụ osisi na-enweghị ọrịa maka ịkụ, họrọ ụdị akpụ nke na-eguzogide, ma wepụ ma bibie osisi ndị metụtara ya ozugbo. Jikwaa ijiji ọcha na ọgwụ mgbochi insects kwesịrị ekwes